# YouTube vs Traditional News (India) - Analysis Notebook

Isme koi setup nahi karna, bas upar se niche har cell ko Shift+Enter dabate jaana.

**Kya hoga:**
1. Folders bana ke mock (realistic fake) data generate karenge
2. Data clean karke engagement metrics nikalenge (like rate, comments/1k views, etc.)
3. Comments pe sentiment analysis chalayenge (positive/negative/neutral)
4. Result CSV files download kar sakte ho, SQL aur Power BI me use karne ke liye

## Step 1: Setup - folders aur libraries

In [2]:
!pip install -q vaderSentiment
import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)
print("Setup done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 5.6 MB/s eta 0:00:00
Setup done


## Step 2: Mock data generate karo

Ye fake but realistic data banayega - 6 channels (3 mainstream news + 3 independent creators), 25 videos har channel ke, aur 20 comments har video ke.

In [1]:
"""
Generates realistic MOCK data in the same shape as api_pull.py's output,
so you can build and test the full pipeline (cleaning, sentiment, SQL,
Power BI) before your YouTube API key/quota is sorted out.

Output:
    data/raw/channels.csv
    data/raw/videos.csv
    data/raw/comments.csv

Once you have a real API key, just run api_pull.py instead - it writes to
the exact same files/columns, so nothing downstream needs to change.
"""

import csv
import os
import random
from datetime import datetime, timedelta, timezone

random.seed(42)

CHANNELS = [
    {"channel_id": "UC_aajtak", "name": "AajTak", "title": "Aaj Tak",
     "channel_type": "mainstream_news", "subscriber_count": 45_200_000,
     "video_count": 189000, "view_count": 32_000_000_000},
    {"channel_id": "UC_ndtv", "name": "NDTV", "title": "NDTV",
     "channel_type": "mainstream_news", "subscriber_count": 12_100_000,
     "video_count": 210000, "view_count": 9_800_000_000},
    {"channel_id": "UC_republic", "name": "RepublicWorld", "title": "Republic World",
     "channel_type": "mainstream_news", "subscriber_count": 9_600_000,
     "video_count": 165000, "view_count": 7_200_000_000},
    {"channel_id": "UC_dhruv", "name": "DhruvRathee", "title": "Dhruv Rathee",
     "channel_type": "independent_creator", "subscriber_count": 26_800_000,
     "video_count": 890, "view_count": 3_900_000_000},
    {"channel_id": "UC_deshbhakt", "name": "TheDeshbhakt", "title": "The Deshbhakt",
     "channel_type": "independent_creator", "subscriber_count": 6_400_000,
     "video_count": 640, "view_count": 980_000_000},
    {"channel_id": "UC_beerbiceps", "name": "RanveerAllahbadia", "title": "BeerBiceps",
     "channel_type": "independent_creator", "subscriber_count": 8_100_000,
     "video_count": 1500, "view_count": 1_600_000_000},
]

TOPICS = [
    "Elections 2026", "Economy & Inflation", "Cricket", "Bollywood",
    "Weather Alert", "Crime News", "Foreign Policy", "Tech & AI",
    "Farmers Protest", "Court Verdict", "Explainer", "Interview",
]

POSITIVE_COMMENTS = [
    "Great reporting, very balanced coverage",
    "Finally someone explaining this clearly",
    "This channel actually does its research",
    "Thanks for the honest update on this",
    "Best explainer I have seen on this topic",
]
NEGATIVE_COMMENTS = [
    "This is so biased, disappointing",
    "Clickbait title, video says nothing new",
    "Stop spreading fear for views",
    "Waste of time, no real information",
    "Anchor is shouting again, unwatchable",
]
NEUTRAL_COMMENTS = [
    "What time did this happen",
    "Can someone summarize this in short",
    "Which state is this news from",
    "First comment",
    "Watching this in 2026",
]


def random_date_within(days_back: int) -> str:
    dt = datetime.now(timezone.utc) - timedelta(
        days=random.randint(0, days_back), hours=random.randint(0, 23)
    )
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def make_videos(channel: dict, n: int) -> list[dict]:
    videos = []
    is_mainstream = channel["channel_type"] == "mainstream_news"
    for i in range(n):
        topic = random.choice(TOPICS)
        views = int(random.gauss(
            800_000 if is_mainstream else 450_000,
            300_000 if is_mainstream else 250_000,
        ))
        views = max(5_000, views)
        # independent creators tend to have higher engagement ratios
        like_ratio = random.uniform(0.015, 0.03) if is_mainstream else random.uniform(0.03, 0.07)
        comment_ratio = random.uniform(0.001, 0.003) if is_mainstream else random.uniform(0.003, 0.008)
        videos.append({
            "video_id": f"{channel['channel_id']}_vid{i}",
            "channel_id": channel["channel_id"],
            "channel_type": channel["channel_type"],
            "title": f"{topic}: {channel['title']} Coverage #{i+1}",
            "published_at": random_date_within(60),
            "category_id": "25",
            "duration": f"PT{random.randint(2, 22)}M{random.randint(0,59)}S",
            "view_count": views,
            "like_count": int(views * like_ratio),
            "comment_count": int(views * comment_ratio),
        })
    return videos


def make_comments(video: dict, n: int) -> list[dict]:
    comments = []
    is_mainstream = video["channel_type"] == "mainstream_news"
    # mainstream news skews slightly more negative/critical in comments
    weights = [0.30, 0.45, 0.25] if is_mainstream else [0.55, 0.20, 0.25]
    for _ in range(n):
        bucket = random.choices(
            [POSITIVE_COMMENTS, NEGATIVE_COMMENTS, NEUTRAL_COMMENTS], weights=weights
        )[0]
        comments.append({
            "video_id": video["video_id"],
            "comment_text": random.choice(bucket),
            "like_count": random.randint(0, 500),
            "published_at": random_date_within(60),
        })
    return comments


def main():
    os.makedirs("data/raw", exist_ok=True)
    all_videos, all_comments = [], []

    for ch in CHANNELS:
        videos = make_videos(ch, n=25)
        all_videos.extend(videos)
        for v in videos:
            all_comments.extend(make_comments(v, n=20))

    _write_csv("data/raw/channels.csv", CHANNELS)
    _write_csv("data/raw/videos.csv", all_videos)
    _write_csv("data/raw/comments.csv", all_comments)

    print(f"Mock data generated: {len(CHANNELS)} channels, "
          f"{len(all_videos)} videos, {len(all_comments)} comments")
    print("Files written to data/raw/. Now run scripts/clean_data.py")


def _write_csv(path: str, rows: list[dict]):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    main()


main()


Mock data generated: 6 channels, 150 videos, 3000 comments
Files written to data/raw/. Now run scripts/clean_data.py
Mock data generated: 6 channels, 150 videos, 3000 comments
Files written to data/raw/. Now run scripts/clean_data.py


## Step 3: Data clean karo

Engagement metrics calculate honge: like rate, comment rate, comments per 1k views, views per day.

In [3]:
"""
Cleans and enriches the raw data pulled by api_pull.py / generate_sample_data.py.

Reads:
    data/raw/channels.csv
    data/raw/videos.csv
    data/raw/comments.csv

Writes:
    data/cleaned/channels_clean.csv
    data/cleaned/videos_clean.csv       (with engagement metrics added)
    data/cleaned/channel_summary.csv    (aggregated stats per channel)
"""

import os
import pandas as pd


def parse_iso_duration(duration: str) -> int:
    """Convert ISO 8601 duration (e.g. PT12M30S) to total seconds."""
    import re
    if not isinstance(duration, str):
        return 0
    match = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", duration)
    if not match:
        return 0
    h, m, s = (int(x) if x else 0 for x in match.groups())
    return h * 3600 + m * 60 + s


def main():
    os.makedirs("data/cleaned", exist_ok=True)

    channels = pd.read_csv("data/raw/channels.csv")
    videos = pd.read_csv("data/raw/videos.csv")

    # --- clean videos ---
    videos["published_at"] = pd.to_datetime(videos["published_at"])
    videos["duration_seconds"] = videos["duration"].apply(parse_iso_duration)
    videos = videos.drop_duplicates(subset="video_id")
    videos = videos[videos["view_count"] > 0]  # drop broken/zero-view rows

    # engagement metrics
    videos["like_rate"] = videos["like_count"] / videos["view_count"]
    videos["comment_rate"] = videos["comment_count"] / videos["view_count"]
    videos["comments_per_1k_views"] = videos["comment_count"] / (videos["view_count"] / 1000)
    videos["days_since_upload"] = (
        pd.Timestamp.now(tz="UTC") - videos["published_at"]
    ).dt.days.clip(lower=1)
    videos["views_per_day"] = videos["view_count"] / videos["days_since_upload"]

    videos.to_csv("data/cleaned/videos_clean.csv", index=False)

    # --- channel-level summary ---
    summary = (
        videos.groupby(["channel_id", "channel_type"])
        .agg(
            videos_analyzed=("video_id", "count"),
            avg_views=("view_count", "mean"),
            avg_like_rate=("like_rate", "mean"),
            avg_comment_rate=("comment_rate", "mean"),
            avg_views_per_day=("views_per_day", "mean"),
        )
        .reset_index()
        .merge(channels[["channel_id", "name", "subscriber_count"]], on="channel_id", how="left")
    )
    summary["avg_like_rate_pct"] = (summary["avg_like_rate"] * 100).round(2)
    summary["avg_comment_rate_pct"] = (summary["avg_comment_rate"] * 100).round(3)
    summary.to_csv("data/cleaned/channel_summary.csv", index=False)

    channels.to_csv("data/cleaned/channels_clean.csv", index=False)

    print("Cleaned data written to data/cleaned/")
    print(summary[["name", "channel_type", "avg_views", "avg_like_rate_pct",
                    "avg_comment_rate_pct"]].to_string(index=False))


if __name__ == "__main__":
    main()


main()


Cleaned data written to data/cleaned/
             name        channel_type  avg_views  avg_like_rate_pct  avg_comment_rate_pct
           AajTak     mainstream_news  750596.48               2.23                 0.211
RanveerAllahbadia independent_creator  487645.76               5.10                 0.560
     TheDeshbhakt independent_creator  505209.20               5.29                 0.546
      DhruvRathee independent_creator  452907.52               4.91                 0.496
             NDTV     mainstream_news  751118.80               2.33                 0.196
    RepublicWorld     mainstream_news  900857.60               2.25                 0.188
Cleaned data written to data/cleaned/
             name        channel_type  avg_views  avg_like_rate_pct  avg_comment_rate_pct
           AajTak     mainstream_news  750596.48               2.23                 0.211
RanveerAllahbadia independent_creator  487645.76               5.10                 0.560
     TheDeshbhakt indepe

## Step 4: Sentiment analysis chalao

Har comment ko positive / negative / neutral me classify karega (VADER model use karke).

In [4]:
"""
Runs VADER sentiment analysis on video comments and joins the result back
to video/channel info so you can compare sentiment across channel types.

Reads:
    data/raw/comments.csv
    data/cleaned/videos_clean.csv

Writes:
    data/cleaned/comments_scored.csv
    data/cleaned/sentiment_by_channel_type.csv
"""

import os
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()


def score_sentiment(text: str) -> tuple[float, str]:
    if not isinstance(text, str) or not text.strip():
        return 0.0, "neutral"
    compound = analyzer.polarity_scores(text)["compound"]
    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"
    return compound, label


def main():
    os.makedirs("data/cleaned", exist_ok=True)

    comments = pd.read_csv("data/raw/comments.csv")
    videos = pd.read_csv("data/cleaned/videos_clean.csv")[
        ["video_id", "channel_id", "channel_type", "title"]
    ]

    comments = comments.merge(videos, on="video_id", how="inner")

    scores = comments["comment_text"].apply(score_sentiment)
    comments["sentiment_score"] = scores.apply(lambda x: x[0])
    comments["sentiment_label"] = scores.apply(lambda x: x[1])

    comments.to_csv("data/cleaned/comments_scored.csv", index=False)

    summary = (
        comments.groupby("channel_type")["sentiment_label"]
        .value_counts(normalize=True)
        .rename("share")
        .reset_index()
        .pivot(index="channel_type", columns="sentiment_label", values="share")
        .fillna(0)
        .round(3)
        .reset_index()
    )
    summary.to_csv("data/cleaned/sentiment_by_channel_type.csv", index=False)

    print("Sentiment scoring complete.")
    print(summary.to_string(index=False))


if __name__ == "__main__":
    main()


main()


Sentiment scoring complete.
       channel_type  negative  neutral  positive
independent_creator     0.112    0.441     0.447
    mainstream_news     0.261    0.488     0.251
Sentiment scoring complete.
       channel_type  negative  neutral  positive
independent_creator     0.112    0.441     0.447
    mainstream_news     0.261    0.488     0.251


## Step 5: Results dekho

In [5]:
import pandas as pd

print("=== Channel Summary ===")
display(pd.read_csv("data/cleaned/channel_summary.csv"))

print("\n=== Sentiment by Channel Type ===")
display(pd.read_csv("data/cleaned/sentiment_by_channel_type.csv"))


=== Channel Summary ===


,channel_id,channel_type,videos_analyzed,avg_views,avg_like_rate,avg_comment_rate,avg_views_per_day,name,subscriber_count,avg_like_rate_pct,avg_comment_rate_pct
0,UC_aajtak,mainstream_news,25,750596.48,0.022334,0.002115,55933.263268,AajTak,45200000,2.23,0.211
1,UC_beerbiceps,independent_creator,25,487645.76,0.051011,0.005602,35135.953605,RanveerAllahbadia,8100000,5.10,0.560
2,UC_deshbhakt,independent_creator,25,505209.20,0.052942,0.005459,85941.259174,TheDeshbhakt,6400000,5.29,0.546
3,UC_dhruv,independent_creator,25,452907.52,0.049052,0.004956,49537.751036,DhruvRathee,26800000,4.91,0.496
4,UC_ndtv,mainstream_news,25,751118.80,0.023305,0.001956,33717.713813,NDTV,12100000,2.33,0.196
5,UC_republic,mainstream_news,25,900857.60,0.022544,0.001883,47534.897320,RepublicWorld,9600000,2.25,0.188



=== Sentiment by Channel Type ===


,channel_type,negative,neutral,positive
0,independent_creator,0.112,0.441,0.447
1,mainstream_news,0.261,0.488,0.251


## Step 6: CSV files download karo

Inhe apne PC pe download kar lo - SQL me load karne ke liye ya Power BI me connect karne ke liye.

In [6]:
from google.colab import files
import shutil

shutil.make_archive("cleaned_data", "zip", "data/cleaned")
files.download("cleaned_data.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Next steps
- Real data chahiye? YouTube Data API v3 key free me lo (Google Cloud Console se), phir api_pull.py wala script isi tarah ek cell me daal ke chala sakte ho - bas apna API key daalna hoga.
- SQL: sql/schema.sql aur sql/analysis_queries.sql (zip me the) in downloaded CSVs ke saath use karo.
- Power BI: downloaded cleaned_data.zip ko extract karke Power BI me Get Data se CSV connect karo.